# Extract cluster → cell_ontology_id mapping

Open an `.h5ad`, pull unique `(cluster_name, cell_ontology_id)` pairs from `adata.obs`, and write a CSV with 4 columns:

`cluster_name, skos, manual_mapped_cid, cell_ontology_id`

`skos` is restricted to `"exact"` or `"related"` (blank by default — fill in manually or set `DEFAULT_SKOS` below).

In [ ]:
import pandas as pd
import scanpy as sc
from pathlib import Path

# ---- edit these ----
H5AD_PATH       = "path/to/your.h5ad"
CLUSTER_HEADER  = "cluster"                           # column in adata.obs
CID_COLUMN      = "cell_type_ontology_term_id"        # column in adata.obs
DEFAULT_SKOS    = ""                                  # "", "exact", or "related"
OUTPUT_CSV      = None                                # None -> auto from H5AD_PATH
# --------------------

ALLOWED_SKOS = {"", "exact", "related"}
assert DEFAULT_SKOS in ALLOWED_SKOS, f"DEFAULT_SKOS must be one of {sorted(ALLOWED_SKOS)}"

In [ ]:
adata = sc.read_h5ad(H5AD_PATH)
print(adata)

for col in (CLUSTER_HEADER, CID_COLUMN):
    if col not in adata.obs.columns:
        raise KeyError(f"{col!r} not found in obs. Available: {list(adata.obs.columns)}")

In [ ]:
pairs = (
    adata.obs[[CLUSTER_HEADER, CID_COLUMN]]
    .astype(str)
    .drop_duplicates()
    .sort_values([CLUSTER_HEADER, CID_COLUMN])
    .reset_index(drop=True)
)

mapping = pd.DataFrame({
    "cluster_name":      pairs[CLUSTER_HEADER],
    "skos":              DEFAULT_SKOS,
    "manual_mapped_cid": "",
    "cell_ontology_id":  pairs[CID_COLUMN],
})

print(f"{pairs[CLUSTER_HEADER].nunique()} unique clusters, {len(mapping)} rows")
mapping.head(20)

In [ ]:
out_path = (
    Path(OUTPUT_CSV)
    if OUTPUT_CSV
    else Path(H5AD_PATH).with_suffix("").with_name(Path(H5AD_PATH).stem + "_cluster_cid_mapping.csv")
)
mapping.to_csv(out_path, index=False)
print(f"Wrote {len(mapping)} rows to {out_path}")